In [5]:
import pandas as pd
import os
from scipy.stats import wilcoxon

# ==============================================================================
# 1. DATA LOADING AND PREPARATION
# ==============================================================================
file_path = 'results/results_HT_delay_1.csv' 
target_horizon = 150

top_10_diversified = [
    'VALE3',  
    'PETR4',  
    'ITUB4',  
    'RENT3',  
    'ELET3',  
    'LREN3',  
    'GGBR4',  
    'HYPE3',  
    'BRFS3',  
    'CIEL3',  
]

domain_map = {'fund': 'Fundamental', 'nao_fund': 'Technical'}

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    if 'fh' in df.columns: 
        df.rename(columns={'fh': 'horizon'}, inplace=True)
    
    df_filtered = df[df['lag'] == target_horizon].copy()
    df_target = df_filtered[df_filtered['dataset'].isin(top_10_diversified)].copy()
    df_target['Domain'] = df_target['category'].map(domain_map)
else:
    raise FileNotFoundError(f"WARNING: File {file_path} not found. Cannot compute statistical test.")

# ==============================================================================
# 2. STATISTICAL SIGNIFICANCE TEST (WILCOXON SIGNED-RANK TEST - KAPPA)
# ==============================================================================
print("\n" + "="*60)
print(f"SIGNIFICANCE TEST (WILCOXON - KAPPA) - h={target_horizon} (TOP 10 ASSETS)")
print("="*60)

try:
    # 1. Group confusion matrix by ASSET and DOMAIN
    df_assets = df_target.groupby(['dataset', 'Domain'])[['TP', 'TN', 'FP', 'FN']].sum().reset_index()
    
    # 2. Calculate Kappa for each asset
    total_assets = df_assets['TP'] + df_assets['TN'] + df_assets['FP'] + df_assets['FN']
    
    # Observed accuracy (po)
    po = (df_assets['TP'] + df_assets['TN']) / total_assets
    
    # Probability of random agreement (pe)
    pe_pos = ((df_assets['TP'] + df_assets['FP']) * (df_assets['TP'] + df_assets['FN'])) / (total_assets**2)
    pe_neg = ((df_assets['TN'] + df_assets['FN']) * (df_assets['TN'] + df_assets['FP'])) / (total_assets**2)
    pe = pe_pos + pe_neg
    
    # Cohen's Kappa
    df_assets['Kappa'] = (po - pe) / (1 - pe)
    
    # 3. Separate Fundamental and Technical data for pairing
    kappa_fund = df_assets[df_assets['Domain'] == 'Fundamental'].set_index('dataset')['Kappa']
    kappa_tech = df_assets[df_assets['Domain'] == 'Technical'].set_index('dataset')['Kappa']
    
    # Ensure perfect alignment by asset (index)
    df_paired = pd.concat([kappa_fund, kappa_tech], axis=1, keys=['Fundamental', 'Technical']).dropna()
    
    # 4. Run Wilcoxon Signed-Rank Test
    # Alternative hypothesis 'greater': testing if Fundamental Kappa is significantly GREATER than Technical
    stat, pval = wilcoxon(df_paired['Fundamental'], df_paired['Technical'], alternative='greater')
    
    print("Paired Kappa by Asset:")
    print(df_paired.to_string(formatters={'Fundamental': '{:.4f}'.format, 'Technical': '{:.4f}'.format}))
    print("-" * 40)
    print(f"Wilcoxon Statistic: {stat:.1f}")
    print(f"P-value:            {pval:.4e}")
    
    if pval < 0.1:
        print("\nConclusion: The superiority of the Fundamental model (Kappa) IS statistically significant (p < 0.1).")
    else:
        print("\nConclusion: The difference IS NOT statistically significant (p >= 0.1).")
        
except Exception as e:
    print(f"Error computing the statistical test: {e}")
print("="*60)


SIGNIFICANCE TEST (WILCOXON - KAPPA) - h=150 (TOP 10 ASSETS)
Paired Kappa by Asset:
        Fundamental Technical
dataset                      
BRFS3       -0.1282   -0.1643
CIEL3        0.2425    0.0051
ELET3        0.0042   -0.0762
GGBR4       -0.0135   -0.1482
HYPE3        0.1191    0.0165
ITUB4       -0.1625   -0.1444
LREN3        0.1725   -0.1300
PETR4       -0.0362   -0.1206
RENT3       -0.2661    0.0517
VALE3       -0.0508   -0.1479
----------------------------------------
Wilcoxon Statistic: 44.0
P-value:            5.2734e-02

Conclusion: The superiority of the Fundamental model (Kappa) IS statistically significant (p < 0.1).
